In [1]:
import numpy as np
import tensorflow as tf

from keras import layers
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from pathlib import Path
from data_loader import load_datasets

In [ ]:
base_path = Path(r"C:.\DeepLOB_T\data")

x_train, y_train, x_val, y_val, x_test, y_test = load_datasets(base_path=base_path, timestamp_per_sample=100)

print(x_train.shape, y_train.shape)
print(x_val.shape, y_val.shape)
print(x_test.shape, y_test.shape)

(203720, 100, 40, 1) (203720, 3)
(50931, 100, 40, 1) (50931, 3)
(139488, 100, 40, 1) (139488, 3)


In [ ]:
# MODEL CNN-LSTM

model_1 = tf.keras.models.load_model(r"C:.\DeepLOB_TE\models\CNN_LSTM_model.keras")



# MODEL CNN-T

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, embedding_dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.embedding_dim = embedding_dim
        self.position_embedding = layers.Embedding(
            input_dim=sequence_length,
            output_dim=embedding_dim
        )

    def call(self, inputs):
        positions = tf.range(start=0, limit=self.sequence_length, delta=1)
        embedded_positions = self.position_embedding(positions)
        return inputs + embedded_positions

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "embedding_dim": self.embedding_dim,
        })
        return config

model_2 = tf.keras.models.load_model(
    r"C:.\DeepLOB_TE\models\CNN_T_model.keras",
    custom_objects={"PositionalEmbedding": PositionalEmbedding}
)

c:\Users\GiovanniPiva\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\layer.py:427: UserWarning: `build()` was called on layer 'positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


In [6]:
test_loss, test_acc = model_1.evaluate(
    x_test,
    y_test,
    batch_size=128,
    verbose=2
)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model_1.predict(x_test, batch_size=128), axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))

1090/1090 - 37s - 34ms/step - accuracy: 0.7758 - loss: 0.5779
Test loss: 0.5779002904891968
Test accuracy: 0.7758445143699646
1090/1090 ━━━━━━━━━━━━━━━━━━━━ 39s 36ms/step
[[26295  6398  5715]
 [ 4263 56513  5220]
 [ 4014  5657 25413]]
              precision    recall  f1-score   support

           0     0.7606    0.6846    0.7206     38408
           1     0.8242    0.8563    0.8399     65996
           2     0.6992    0.7243    0.7115     35084

    accuracy                         0.7758    139488
   macro avg     0.7613    0.7551    0.7574    139488
weighted avg     0.7752    0.7758    0.7748    139488



In [5]:
test_loss, test_acc = model_2.evaluate(
    x_test,
    y_test,
    batch_size=128,
    verbose=2
)

print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model_2.predict(x_test, batch_size=128), axis=1)

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, digits=4))

1090/1090 - 84s - 77ms/step - accuracy: 0.7478 - loss: 0.6262
Test loss: 0.6261819005012512
Test accuracy: 0.7478277683258057
1090/1090 ━━━━━━━━━━━━━━━━━━━━ 85s 77ms/step
[[25658  6736  6014]
 [ 5669 55521  4806]
 [ 5854  6096 23134]]
              precision    recall  f1-score   support

           0     0.6901    0.6680    0.6789     38408
           1     0.8123    0.8413    0.8265     65996
           2     0.6813    0.6594    0.6702     35084

    accuracy                         0.7478    139488
   macro avg     0.7279    0.7229    0.7252    139488
weighted avg     0.7457    0.7478    0.7465    139488

